In [ ]:
# ===================================================================
# 2D human pose: Keypoint R-CNN scored on the COCO val2017 keypoints
# split it was never trained on.
#
# Honest framing: the weights are torchvision's published
# KeypointRCNN_ResNet50_FPN COCO_V1, trained on COCO train2017.
# val2017 is held out from that training. Nothing is fine-tuned here;
# what this notebook contributes is the measurement, including the
# per-joint breakdown that aggregate AP hides.
# ===================================================================
import os, sys, json, glob, time, collections
print("python", sys.version)
import torch, torchvision
print("torch", torch.__version__, "torchvision", torchvision.__version__, "cuda", torch.cuda.is_available())

ROOT = "/kaggle/input"
print("TOP LEVEL:", sorted(os.listdir(ROOT)))

# Full walk, NO depth cap, real basenames printed.
ext = collections.Counter(); bydir = collections.Counter(); samples = []
ndirs = 0
for dp, dn, fn in os.walk(ROOT):
    ndirs += 1
    if fn:
        bydir[os.path.relpath(dp, ROOT)] += len(fn)
    for f in fn:
        ext[os.path.splitext(f)[1].lower()] += 1
        if len(samples) < 25:
            samples.append(os.path.join(os.path.relpath(dp, ROOT), f))
print("dirs walked:", ndirs)
print("extensions:", ext.most_common(10))
print("top dirs:")
for d, c in bydir.most_common(20):
    print("  %8d %s" % (c, d))
print("sample real paths:")
for s in samples:
    print("   ", s)

In [ ]:
# ---- locate the held-out annotation file and its images ----
ann_cands = glob.glob(ROOT + "/**/person_keypoints_val2017.json", recursive=True)
assert ann_cands, "no person_keypoints_val2017.json under /kaggle/input"
ANN = ann_cands[0]
print("annotations:", ANN)

img_dirs = [p for p in glob.glob(ROOT + "/**/val2017", recursive=True) if os.path.isdir(p)]
print("candidate val2017 image dirs:", img_dirs)
assert img_dirs, "no val2017 image directory found"

from pycocotools.coco import COCO
coco = COCO(ANN)
person_cat = coco.getCatIds(catNms=["person"])
print("person cat ids:", person_cat)

IMG_DIR = None
best = -1
for d in img_dirs:
    names = set(os.listdir(d))
    n = sum(1 for i in coco.loadImgs(coco.getImgIds()) if i["file_name"] in names)
    print("  %s: %d / %d annotated images present" % (d, n, len(coco.getImgIds())))
    if n > best:
        best, IMG_DIR = n, d
print("using IMG_DIR:", IMG_DIR, "with", best, "images present")
assert best >= 2000, "only %d val2017 images on disk; cannot score the held-out split" % best

on_disk = set(os.listdir(IMG_DIR))
eval_img_ids = []
for iid in sorted(coco.getImgIds(catIds=person_cat)):
    info = coco.loadImgs(iid)[0]
    if info["file_name"] not in on_disk:
        continue
    anns = coco.loadAnns(coco.getAnnIds(imgIds=iid, catIds=person_cat, iscrowd=False))
    if any(a.get("num_keypoints", 0) > 0 for a in anns):
        eval_img_ids.append(iid)
print("IMAGES TO SCORE (have >=1 keypoint-annotated person):", len(eval_img_ids))
assert len(eval_img_ids) >= 1000, "too few annotated images to report a number"

KP_NAMES = ["nose", "left_eye", "right_eye", "left_ear", "right_ear",
            "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
            "left_wrist", "right_wrist", "left_hip", "right_hip",
            "left_knee", "right_knee", "left_ankle", "right_ankle"]
assert coco.loadCats(person_cat)[0]["keypoints"] == KP_NAMES, coco.loadCats(person_cat)[0]["keypoints"]

In [ ]:
# ---- model ----
from torchvision.models.detection import keypointrcnn_resnet50_fpn, KeypointRCNN_ResNet50_FPN_Weights
dev = "cuda" if torch.cuda.is_available() else "cpu"
assert dev == "cuda", "GPU not attached"
W = KeypointRCNN_ResNet50_FPN_Weights.COCO_V1
model = keypointrcnn_resnet50_fpn(weights=W).eval().to(dev)
print("weights:", W)
print("published metrics from weights meta:", W.meta.get("_metrics"))

from PIL import Image
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

class ValImgs(Dataset):
    def __init__(self, ids):
        self.ids = ids
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        iid = self.ids[i]
        info = coco.loadImgs(iid)[0]
        im = Image.open(os.path.join(IMG_DIR, info["file_name"])).convert("RGB")
        return TF.to_tensor(im), iid

def collate(b):
    return [x[0] for x in b], [x[1] for x in b]

dl = DataLoader(ValImgs(eval_img_ids), batch_size=4, num_workers=2, collate_fn=collate)

results = []
preds_by_img = {}
t0 = time.time()
with torch.inference_mode():
    for bi, (imgs, ids) in enumerate(dl):
        outs = model([im.to(dev) for im in imgs])
        for iid, o in zip(ids, outs):
            keep = o["scores"] > 0.05
            boxes = o["boxes"][keep].cpu().numpy()
            kps = o["keypoints"][keep].cpu().numpy()
            scores = o["scores"][keep].cpu().numpy()
            preds_by_img[iid] = [(boxes[i], kps[i], float(scores[i])) for i in range(len(scores))]
            for i in range(len(scores)):
                flat = []
                for j in range(17):
                    flat += [float(kps[i, j, 0]), float(kps[i, j, 1]), 1.0]
                results.append({"image_id": int(iid), "category_id": 1,
                                "keypoints": flat, "score": float(scores[i])})
        if bi % 100 == 0:
            print("  batch %d/%d  %.0fs" % (bi, len(dl), time.time() - t0), flush=True)
print("inference done in", round(time.time() - t0, 1), "s; detections:", len(results))
assert len(results) > 0, "model produced no detections at all"
print("images with >=1 detection:", sum(1 for v in preds_by_img.values() if v), "/", len(preds_by_img))

In [ ]:
# ---- OKS-based AP via pycocotools (the official protocol) ----
from pycocotools.cocoeval import COCOeval
RES = "/kaggle/working/kp_results.json"
json.dump(results, open(RES, "w"))
cocoDt = coco.loadRes(RES)
E = COCOeval(coco, cocoDt, "keypoints")
E.params.imgIds = eval_img_ids
E.evaluate(); E.accumulate(); E.summarize()
AP_NAMES = ["AP", "AP50", "AP75", "AP_medium", "AP_large", "AR", "AR50", "AR75", "AR_medium", "AR_large"]
oks = dict((n, round(float(v), 5)) for n, v in zip(AP_NAMES, E.stats))
print("OKS AP block:", json.dumps(oks, indent=1))
assert oks["AP"] > 0.1, "OKS AP %s is implausibly low - evaluation is broken, not the model" % oks["AP"]

In [ ]:
# ---- PCK: match detections to GT persons by box IoU, then per-joint ----
import numpy as np

def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    iw = max(0.0, x2 - x1); ih = max(0.0, y2 - y1)
    inter = iw * ih
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

THRESH = [0.05, 0.1, 0.2]
hit = dict((t, np.zeros(17)) for t in THRESH)
tot = np.zeros(17)
dists = [[] for _ in range(17)]
n_gt = 0
n_matched = 0
unmatched_gt = 0
BOX_IOU_MIN = 0.5

for iid in eval_img_ids:
    gts = [a for a in coco.loadAnns(coco.getAnnIds(imgIds=iid, catIds=person_cat, iscrowd=False))
           if a.get("num_keypoints", 0) > 0]
    dets = sorted(preds_by_img.get(iid, []), key=lambda d: -d[2])
    used = set()
    for g in gts:
        n_gt += 1
        x, y, w, h = g["bbox"]
        gbox = [x, y, x + w, y + h]
        bi, bv = -1, 0.0
        for i, (dbox, dk, ds) in enumerate(dets):
            if i in used:
                continue
            v = iou_xyxy(gbox, dbox)
            if v > bv:
                bi, bv = i, v
        if bi < 0 or bv < BOX_IOU_MIN:
            unmatched_gt += 1
            continue
        used.add(bi)
        n_matched += 1
        dk = dets[bi][1]
        norm = max(w, h)
        if norm <= 0:
            continue
        gk = np.array(g["keypoints"], dtype=np.float64).reshape(17, 3)
        for j in range(17):
            if gk[j, 2] <= 0:
                continue
            tot[j] += 1
            d = float(np.hypot(dk[j, 0] - gk[j, 0], dk[j, 1] - gk[j, 1]) / norm)
            dists[j].append(d)
            for t in THRESH:
                if d <= t:
                    hit[t][j] += 1

print("GT persons:", n_gt, "| matched (box IoU>=0.5):", n_matched, "| unmatched:", unmatched_gt)
assert n_matched >= 500, "only %d matched person pairs - nothing was really scored" % n_matched
assert tot.sum() >= 5000, "only %d labelled joints scored" % int(tot.sum())

pck = {}
for t in THRESH:
    overall = float(hit[t].sum() / tot.sum())
    perj = dict((KP_NAMES[j], round(float(hit[t][j] / tot[j]), 5) if tot[j] else None) for j in range(17))
    pck["PCK@%s" % t] = {"overall_on_matched": round(overall, 5), "per_joint": perj}
    print("\nPCK@%s  overall (matched persons) = %.5f" % (t, overall))
    for j in sorted(range(17), key=lambda j: hit[t][j] / max(tot[j], 1)):
        print("   %15s %.4f   (n=%d)" % (KP_NAMES[j], hit[t][j] / max(tot[j], 1), int(tot[j])))

gt_joint_total = 0
for iid in eval_img_ids:
    for a in coco.loadAnns(coco.getAnnIds(imgIds=iid, catIds=person_cat, iscrowd=False)):
        if a.get("num_keypoints", 0) > 0:
            gt_joint_total += int((np.array(a["keypoints"]).reshape(17, 3)[:, 2] > 0).sum())
for t in THRESH:
    pck["PCK@%s" % t]["overall_all_gt_persons"] = round(float(hit[t].sum() / gt_joint_total), 5)
print("\nlabelled joints on matched persons:", int(tot.sum()), "| on all GT persons:", gt_joint_total)
print("PCK@0.2 incl. missed persons:", pck["PCK@0.2"]["overall_all_gt_persons"])

median_err = dict((KP_NAMES[j], round(float(np.median(dists[j])), 5) if dists[j] else None) for j in range(17))
print("median normalised error per joint:", json.dumps(median_err, indent=1))

# upper-body vs extremity summary
EXTREM = ["left_wrist", "right_wrist", "left_ankle", "right_ankle"]
CORE = ["left_shoulder", "right_shoulder", "left_hip", "right_hip", "nose"]
def grp(names, t):
    h = sum(hit[t][KP_NAMES.index(n)] for n in names)
    s = sum(tot[KP_NAMES.index(n)] for n in names)
    return round(float(h / s), 5), int(s)
g_ext = grp(EXTREM, 0.2)
g_core = grp(CORE, 0.2)
print("PCK@0.2 extremities (wrists+ankles):", g_ext, " core (shoulders+hips+nose):", g_core)

In [ ]:
# ---- BAR CHECK: judge this run against the published number for the same
# ---- model on the same benchmark. A number far below the bar is a bug
# ---- report about this pipeline, not a finding about the model.
BAR_PASS, BAR_BUG, BAR_REF = 0.60, 0.50, "Keypoint R-CNN COCO_V1 official COCO keypoint AP ~0.65"
got = oks["AP"]
verdict = "PASS" if got >= BAR_PASS else ("BUG - DIAGNOSE, DO NOT PUBLISH" if got < BAR_BUG else "BELOW BAR")
print("=" * 64)
print("BAR: OKS AP >= %.2f   (reference: %s)" % (BAR_PASS, BAR_REF))
print("GOT: OKS AP  = %.5f" % got)
print("VERDICT: %s" % verdict)
if got < BAR_BUG:
    print("An AP this low points at a decoding, normalisation or flip bug in THIS notebook.")
print("=" * 64)

out = {
    "task": "2D human pose estimation on COCO val2017 keypoints",
    "model": "torchvision keypointrcnn_resnet50_fpn, KeypointRCNN_ResNet50_FPN_Weights.COCO_V1, as published, no fine-tuning",
    "published_metrics_from_weights_meta": W.meta.get("_metrics"),
    "split": "COCO val2017 - held out from the weights' train2017 training set",
    "n_images_scored": len(eval_img_ids),
    "n_gt_persons": n_gt,
    "n_gt_persons_matched": n_matched,
    "n_gt_persons_unmatched": unmatched_gt,
    "n_labelled_joints_scored": int(tot.sum()),
    "n_labelled_joints_all_gt": int(gt_joint_total),
    "oks_ap": oks,
    "pck": pck,
    "pck_definition": "correct if ||pred-gt|| <= a * max(gt_box_w, gt_box_h); only keypoints with COCO visibility flag v>0 are scored",
    "match_rule": "greedy box-IoU, threshold 0.5, detections ordered by score",
    "median_normalised_error_per_joint": median_err,
    "pck02_extremities_wrists_ankles": {"pck": g_ext[0], "n_joints": g_ext[1]},
    "pck02_core_shoulders_hips_nose": {"pck": g_core[0], "n_joints": g_core[1]},
    "what_this_does_not_show": [
        "One model on one dataset. It says nothing about how this model ranks against any other.",
        "The scores belong to the published weights, not to any training done here.",
        "COCO val2017 is held out from training but is the benchmark this model family was developed against, so it is not an out-of-distribution test.",
        "PCK depends on the normaliser; max(w,h) of the GT box is stated above because the same predictions score differently under torso- or head-normalised variants.",
    ],
}
json.dump(out, open("/kaggle/working/results.json", "w"), indent=1)
print(json.dumps(out, indent=1))
print("\nWROTE /kaggle/working/results.json")